In [1]:
# Hacer peticiones HTTP (GET,POST, ETC)
import requests
# Parsear el HTML y extraer datos
from bs4 import BeautifulSoup
# trabajar con base de datos
import sqlite3
# Unir URLs relativas con la URL Base
from urllib.parse import urljoin
# Para poder interactuar con el sistema operativo
import os
# Generar valores aleatorios
import random
# Manejar pausas
import time

# CONSTANTES GLOBALES 
# URL Base del sitio que se va a scrapear
BASE_URL= "https://books.toscrape.com/"
# Nombre del archivo de la base de datos SQLite donde se guardan los datos
DB_PATH = "books.db"

# Session HTTP
session = requests.Session()

# Mapeo de calificacion
rating_map = {
    "One": 1, "Two": 2, "Three": 3,
    "Four": 4, "Five": 5
}

# Centraliza el scraping
def get_soup(url):
    # Peticion HTTP 
    r = session.get(url, timeout=10) #buscar
    # Manejo de errores
    r.raise_for_status()
    # Devuelve un objeto soup   
    return BeautifulSoup(r.text, "html.parser")

In [2]:
def scrape_book(book_url):
    # Llama a la funcion para convertirlo en un objeto beautiful soup
    soup = get_soup(book_url)

    # Busca la primera etiqueta H1 y extrae solamente el titulo
    title = soup.find("h1").text

    # Busca un <p> con clase price color y extrae el precio
    price = soup.find("p", class_="price_color").text
    price = price.replace("£", "").replace("Â", "")
    price = float(price)

    # Extraer calificacion
    rating = None
    # La calificacion esta en en las clases por eso busca el <p>
    rating_tag = soup.find("p", class_="star-rating")
    # Verificar si el elemento existe 
    if rating_tag:
        # Rating_tag["class"] devuelve una lista donde toma la segunda clase [1]
        rating_class= rating_tag["class"][1]
        #Usa el diccionario y convierte a texto y a numero
        rating= rating_map[rating_class]

    description = None
    # Busca el elemento con id="product_description"
    desc = soup.find(id= "product_description")
    # Verifica que exista antes de continuar
    if desc:
        # Busca la primera <p> despues del div para extraer la descripcion del libro
        description = desc.find_next("p").text

    # Busca la lista del breadcrumb, busca todos los <a> toma el tercer elemento [2] y extrae el texo
    category = soup.select("ul.breadcrumb li")[2].text.strip()
    

    return{
        "title": title,
        "price": price,
        "rating": rating,
        "description": description,
        "category": category, 
        "url": book_url
    }


In [3]:
def scrape_all_books():
    # Obtenemos todo el HTML para sacar las categorias
    soup = get_soup(BASE_URL)
    # Lista vacia para Categorias
    categorias = []

    # Buscar los links de las cateogorias
    links = soup.select(".side_categories ul li ul li a")

    # Recorrer categoria 
    for a in links: 
        # Obtener nombre de la categoria, extra todo el texto del <a> y con .strip limpia los espacios
        name= a.text.strip()
        # Obtener URL de la categoria 
        url = urljoin(BASE_URL, a["href"]) #href es una URL relativa y URL join la convierte en URL completa
        # Guardar la categoria 
        categorias.append((name, url))

    # Aca se guardan todos los libros
    all_books= []

    # Recorrer categorias 
    for cat in categorias: 
        # Cat es una tupla erntonces lo separamos
        cat_nombre, cat_url = cat
        # Empezar desde la primera pagina
        sig_pag= cat_url

        # Loop para que este scrapeando pagina por pagina
        while sig_pag is not None:
            print("Scrapeando:", sig_pag)
            # Descarga la pagina de la categoria
            s = get_soup(sig_pag)

            # Buscar libros de la pagina
            books = s.select("article.product_pod")

            # Recorrer libros
            for art in books:
                link = art.select_one("h3 a")["href"]
                book_url = urljoin(sig_pag, link)

                # Scrapear el libro
                data= scrape_book(book_url)
                all_books.append(data)
            
            # Buscar boton next
            sig_tbn = s.select_one("li.next a")
            # Si existe siguiente pagina
            if sig_tbn:
                sig_pag = urljoin(sig_pag, sig_tbn["href"])
            else:
                sig_pag = None

    return all_books
books = scrape_all_books()
print("Total libros:", len(books))



Scrapeando: https://books.toscrape.com/catalogue/category/books/travel_2/index.html
Scrapeando: https://books.toscrape.com/catalogue/category/books/mystery_3/index.html
Scrapeando: https://books.toscrape.com/catalogue/category/books/mystery_3/page-2.html
Scrapeando: https://books.toscrape.com/catalogue/category/books/historical-fiction_4/index.html
Scrapeando: https://books.toscrape.com/catalogue/category/books/historical-fiction_4/page-2.html
Scrapeando: https://books.toscrape.com/catalogue/category/books/sequential-art_5/index.html
Scrapeando: https://books.toscrape.com/catalogue/category/books/sequential-art_5/page-2.html
Scrapeando: https://books.toscrape.com/catalogue/category/books/sequential-art_5/page-3.html
Scrapeando: https://books.toscrape.com/catalogue/category/books/sequential-art_5/page-4.html
Scrapeando: https://books.toscrape.com/catalogue/category/books/classics_6/index.html
Scrapeando: https://books.toscrape.com/catalogue/category/books/philosophy_7/index.html
Scrapea

In [4]:
if os.path.exists(DB_PATH):
    os.remove(DB_PATH)

conn = sqlite3.connect(DB_PATH)
cur = conn.cursor()

cur.executescript("""
PRAGMA foreign_keys = ON;

CREATE TABLE categories(
    id INTEGER PRIMARY KEY AUTOINCREMENT,
    name TEXT UNIQUE
);

CREATE TABLE books(
    id INTEGER PRIMARY KEY AUTOINCREMENT,
    title TEXT,
    price REAL,
    rating INTEGER,
    description TEXT,
    category_id INTEGER,
    url TEXT UNIQUE,
    FOREIGN KEY (category_id) REFERENCES categories(id)
);

CREATE TABLE authors(
    id INTEGER PRIMARY KEY AUTOINCREMENT,
    name TEXT UNIQUE
);

CREATE TABLE books_authors(
    book_id INTEGER,
    author_id INTEGER,
    PRIMARY KEY(book_id, author_id),
    FOREIGN KEY(book_id) REFERENCES books(id),
    FOREIGN KEY(author_id) REFERENCES authors(id)
);
""")

conn.commit()
print("Base creada.")

Base creada.


In [6]:
ALL_AUTHORS = [
    "Ana Gómez",
    "Juan Torres",
    "María Ruiz",
    "Carlos Fernández",
    "Lucía López",
    "Diego Cabrera",
    "Sofía Rojas",
    "Andrés Benítez",
    "Marta Acosta",
    "Pablo Vega"
]

def generate_authors_for_book():
    num = random.choice([1, 2])
    return random.sample(ALL_AUTHORS, num)



In [7]:
for b in books:
    categoria = b["category"]

    cur.execute(
        "INSERT OR IGNORE INTO categories(name) VALUES(?)",
        (categoria,)
    )

    cur.execute(
        "SELECT id FROM categories WHERE name=?",
        (categoria,)
    )
    category_id = cur.fetchone()[0]

    cur.execute("""
        INSERT OR IGNORE INTO books
        (title, price, rating, description, category_id, url)
        VALUES (?, ?, ?, ?, ?, ?)
    """, (
        b["title"],
        b["price"],
        b["rating"],
        b["description"],
        category_id,
        b["url"]
    ))

    cur.execute(
        "SELECT id FROM books WHERE url=?",
        (b["url"],)
    )
    book_id = cur.fetchone()[0]

    # ✅ FIX: no pisar ALL_AUTHORS
    book_authors = generate_authors_for_book()

    for author in book_authors:
        cur.execute(
            "INSERT OR IGNORE INTO authors(name) VALUES(?)",
            (author,)
        )

        cur.execute(
            "SELECT id FROM authors WHERE name=?",
            (author,)
        )
        author_id = cur.fetchone()[0]

        cur.execute("""
            INSERT OR IGNORE INTO books_authors
            (book_id, author_id)
            VALUES (?, ?)
        """, (book_id, author_id))

conn.commit()
print("Datos insertados correctamente.")



Datos insertados correctamente.


In [8]:
queries = {
    "1) ¿Autor con más libros de 1 estrella?":
        """
        SELECT a.name, COUNT(*) AS total
        FROM authors a
        JOIN books_authors ba ON a.id = ba.author_id
        JOIN books b ON b.id = ba.book_id
        WHERE b.rating = 1
        GROUP BY a.name
        ORDER BY total DESC
        LIMIT 1;
        """,

    "2) Libros baratos (>3 estrellas y <£10)":
        """
        SELECT title, price, rating
        FROM books
        WHERE rating >= 4 AND price < 10
        ORDER BY price ASC;
        """,

    "3) Categoría con más libros carísimos (>£40)":
        """
        SELECT c.name, COUNT(*) AS total
        FROM books 
        JOIN categories c ON c.id = books.category_id
        WHERE books.price > 40
        GROUP BY c.name
        ORDER BY total DESC
        LIMIT 10;
        """,

    "4) Top 5 libros mejor rankeados pero más baratos":
        """
        SELECT title, rating, price
        FROM books
        ORDER BY rating DESC, price ASC
        LIMIT 5;
        """,

    "5) Cantidad promedio de libros por autor":
        """
        SELECT AVG(cnt)
        FROM (
            SELECT COUNT(*) AS cnt
            FROM books_authors
            GROUP BY author_id
        );
        """
}

for desc, q in queries.items():
    print("\n------", desc, "------")
    for row in cur.execute(q):
        print(row)


------ 1) ¿Autor con más libros de 1 estrella? ------
('Pablo Vega', 45)

------ 2) Libros baratos (>3 estrellas y <£10) ------

------ 3) Categoría con más libros carísimos (>£40) ------
('Default', 59)
('Nonfiction', 40)
('Fiction', 32)
('Sequential Art', 29)
('Add a comment', 28)
('Fantasy', 26)
('Young Adult', 25)
('Romance', 11)
('Mystery', 11)
('Womens Fiction', 10)

------ 4) Top 5 libros mejor rankeados pero más baratos ------
('An Abundance of Katherines', 5, 10.0)
('Greek Mythic History', 5, 10.23)
('The Power Greens Cookbook: 140 Delicious Superfood Recipes', 5, 11.05)
('Dear Mr. Knightley', 5, 11.21)
('The Darkest Corners', 5, 11.33)

------ 5) Cantidad promedio de libros por autor ------
(149.6,)


In [9]:
# Consulta lenta: buscar libros por precio sin índice
query = "SELECT * FROM books WHERE price > 30;"

start = time.time()
cur.execute(query).fetchall()
t1 = time.time() - start

print("Tiempo SIN índice:", t1, "segundos")

# Crear índice
cur.execute("CREATE INDEX idx_books_price ON books(price);")
conn.commit()

start = time.time()
cur.execute(query).fetchall()
t2 = time.time() - start

print("Tiempo CON índice:", t2, "segundos")

Tiempo SIN índice: 0.0032320022583007812 segundos
Tiempo CON índice: 0.002313375473022461 segundos
